In [1]:
using GeneRegulatorySystems
using GeneRegulatorySystems.Models
using GeneRegulatorySystems.Models.V1
using GeneRegulatorySystems.Scheduling
import JSON
using GarishPrint

macro pp(x) :(GarishPrint.pprint($(esc(x)))) end;

Load schedule

In [2]:
path_diff = "examples/specification/differentiation.schedule.json"
path_v1 = "examples/toy/ACDC.schedule.json"
path_kronecker = "examples/benchmark/kronecker-small.schedule.json"
path_rand_diff = "examples/specification/random-differentiation.schedule.json"

schedule! = Models.load(path_rand_diff, seed="seed123");

In [3]:
f! = Scheduling.reify(schedule!, "+.do");

In [88]:
definition = f!.model.model.model.definition;

In [97]:
desc = Models.describe(definition)
components = Dict(typeof(d) => d for d in desc.descriptions)
@pp components[Models.Network].links

87-element Array{@NamedTuple{to::Symbol, from::Symbol, kind::Symbol, properties::Dict{Symbol, Float64}}, 1}:
 (to = Symbol("02"), from = :differentiator01, kind = :activation, properties = Dict(:k => -1.0, :at => 3.8507511580113727))
 (to = Symbol("02"), from = Symbol("02"), kind = :repression, properties = Dict(:k => -1.0, :at => 13.938421661112253))
 (to = Symbol("02"), from = :differentiator10, kind = :repression, properties = Dict(:k => -1.0, :at => 55.094736616011204))
 (to = Symbol("02"), from = :differentiator01, kind = :repression, properties = Dict(:k => -1.0, :at => 7.863090910219008))
 (to = :differentiator01, from = :differentiator01, kind = :activation, properties = Dict(:k => -1.0, :at => 2.0))
 (to = :differentiator01, from = :differentiator00, kind = :proteolysis, properties = Dict(:k => 9.998382791903378e-5))
 (to = :differentiator01, from = :differentiator0_timer, kind = :proteolysis, properties = Dict(:k => 0.0001))
 (to = :differentiator111, from = :differentiator11

Reify the schedule. Need to do a dryrun to get all the primitives

In [ ]:
# this only extracts the first network that it finds in the schedule, rather than all of them
function extract_network(schedule::Scheduling.Schedule)
    network = nothing
    
    function dryrun_collector(primitive!, x, Δt; path, _...)
        if !(isfinite(Δt) && Δt > 0.0)
            return
        end
        if network === nothing
            network = Models.NetworkCreation.entity(primitive!)
        end
    end
    
    schedule(Models.FlatState(); dryrun=dryrun_collector)
    return network
end

network = extract_network(schedule!)